# Phase 1: Environment Setup and Image Processing Utilities

**Objective:** To establish a robust computer vision pipeline for isolating signatures from noisy backgrounds and to determine the baseline Cosine Similarity threshold using a pre-trained ResNet18 model.

This section initializes the necessary libraries, defines the directory structures, and sets up helper functions for consistent image visualization.

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms

os.makedirs('../data/processed/asli_master', exist_ok=True)

def display_image(title, img, target_width=800):
    h, w = img.shape[:2]
    ratio = target_width / float(w)
    dim = (target_width, int(h * ratio))
    resized_img = cv2.resize(img, dim, interpolation=cv2.INTER_AREA)
    cv2.imshow(title, resized_img)

print("Environment setup complete. Libraries loaded.")

✅ Setup Selesai. Semua library siap digunakan.


### 1. Deep Learning Backbone Initialization

We load a pre-trained ResNet18 model to serve as the baseline feature extractor. By detaching the final classification layer, the model is repurposed to output a 512-dimensional continuous feature vector (Signature DNA) which will be used to measure structural similarities.

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)
resnet = torch.nn.Sequential(*(list(resnet.children())[:-1]))
resnet.eval()

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("ResNet18 feature extractor initialized.")

✅ Detektif ResNet18 sudah standby di Lab.


### 2. Feature Extraction Pipeline

This core function processes an input image path through our computer vision pipeline. It applies adaptive Gaussian thresholding and contour detection to aggressively isolate pen strokes from varied paper textures. The cropped region is then padded and passed through the ResNet18 backbone to generate the feature vector.

In [ ]:
def extract_dna_from_path(file_path):
    image = cv2.imread(file_path)
    if image is None: return torch.zeros(512)
        
    # Preprocessing: Grayscale & Adaptive Thresholding
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                    cv2.THRESH_BINARY_INV, 11, 2)
    
    # Contour detection and bounding box extraction with padding
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)
        pad = 15
        x_p, y_p = max(0, x-pad), max(0, y-pad)
        w_p, h_p = min(image.shape[1]-x_p, w+(pad*2)), min(image.shape[0]-y_p, h+(pad*2))
        cropped = image[y_p:y_p+h_p, x_p:x_p+w_p]
    else:
        cropped = image

    # Feature extraction via ResNet18
    sig_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
    input_tensor = preprocess(sig_rgb).unsqueeze(0)
    with torch.no_grad():
        dna = resnet(input_tensor).squeeze()
    return dna

print("Function 'extract_dna_from_path' is ready.")

✅ Fungsi 'extract_dna_from_path' siap beroperasi.


### 3. Automated Signature Harvesting

To build our ground-truth dataset, we apply the aforementioned contour detection logic to extract individual signatures from a raw, multi-signature scanned document. Dimensional filters are applied to reject artifacts, and the valid signatures are saved to the master directory.

In [ ]:
# Define path to the raw multi-signature document
image_path = '../data/raw/sample_ttd2.jpg' 
image = cv2.imread(image_path)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
binary = cv2.adaptiveThreshold(cv2.GaussianBlur(gray, (5, 5), 0), 255, 
                                cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)

contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
image_with_box = image.copy()
count = 0

for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
    # Dimensional filtering to exclude artifacts and edge noise
    if (x > 15 and y > 15 and (x+w) < (image.shape[1]-15)) and (w > 100 and h > 40):
        if cv2.contourArea(cnt) > 2000:
            count += 1
            roi = image[max(0, y-20):min(image.shape[0], y+h+20), max(0, x-20):min(image.shape[1], x+w+20)]
            cv2.imwrite(f'../data/processed/asli_master/asli_{count}.jpg', roi)
            cv2.rectangle(image_with_box, (x-20, y-20), (x+w+20, y+h+20), (0, 255, 0), 4)

display_image('Harvested Signatures', image_with_box)
print(f"Successfully harvested {count} signatures into 'asli_master' directory.")
cv2.waitKey(0)
cv2.destroyAllWindows()

✅ Berhasil memanen 10 tanda tangan ke folder 'asli_master'.


### 4. Forensic Baseline Audit and Threshold Calibration

We evaluate the baseline model's performance by calculating the Cosine Similarity between a master genuine signature and two distinct test groups: genuine variations and unseen external signatures (forgeries/others). The statistical distribution generated here dictates the strict verification margin required for the final production API.

In [ ]:
asli_files = glob.glob('../data/processed/asli_master/*.jpg')
kaggle_files = glob.glob(os.path.join('../data/raw/signatures', "**/*.png"), recursive=True)[:20]

results = []
anchor_dna = extract_dna_from_path(asli_files[0])

# Compare Genuine vs. Genuine
for i in range(1, len(asli_files)):
    test_dna = extract_dna_from_path(asli_files[i])
    score = F.cosine_similarity(anchor_dna.unsqueeze(0), test_dna.unsqueeze(0)).item()
    results.append({'Type': 'Genuine vs Genuine', 'Score': score})

# Compare Genuine vs. Forged/Unseen (Kaggle dataset)
for f in kaggle_files:
    test_dna = extract_dna_from_path(f)
    score = F.cosine_similarity(anchor_dna.unsqueeze(0), test_dna.unsqueeze(0)).item()
    results.append({'Type': 'Genuine vs Forged', 'Score': score})

df = pd.DataFrame(results)
print("\n FORENSIC AUDIT SUMMARY:")
print(df.groupby('Type')['Score'].describe()[['mean', 'min', 'max']])


📊 RINGKASAN AUDIT FORENSIK:
                    mean       min       max
tipe                                        
Asli vs Asli    0.838170  0.798260  0.875526
Asli vs Kaggle  0.616355  0.574441  0.663944
